# Surfactant graph surrogates

Inspect the actual graph/model interfaces and optionally run the recorded pC20 and two-property training workflow.

Install the package from the repository root with `python -m pip install -e ".[research,notebooks]"`. All algorithms are imported from `src/latent_materials/`.

In [ ]:
from pathlib import Path
import pandas as pd
from latent_materials.cli import run_experiment

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
DATA = ROOT / "data"
RESULTS = ROOT / "results"


Build a molecular graph and inspect the two-output mean/log-variance prediction. This uses an untrained model and demonstrates the interface only.

In [ ]:
import torch
from torch_geometric.data import Batch
from latent_materials.surfactants.models import smiles_to_graph, GINHetero
torch.manual_seed(42)
graphs = [smiles_to_graph(s, [0.0, 1.0]) for s in ["CCO", "CCN"]]
model = GINHetero(graphs[0].x.shape[1], 2, layer_sizes=(8, 8), dropout=0.0, head_dim=8).eval()
with torch.no_grad():
    mean, log_variance = model(Batch.from_data_list(graphs))
mean.shape, log_variance.shape

The full workflow prepares the historical split, imputes pC20, constructs the fitted feature columns, searches configurations, trains the two-target surrogate and exports results. Read the README cohort caveats before treating a new run as manuscript verification.

In [ ]:
RUN_FULL_TRAINING = False
if RUN_FULL_TRAINING:
    run_experiment("surfactant_surrogates", data_root=DATA, output=RESULTS / "surfactants/surrogate_training")

## Conditional generation

The generator builds a constrained EO/PO library, trains a structure-only surrogate, applies soft property guidance, fits the conditional adapter and expands candidates through tested mutations. Supply the generated `data/surfpro_train80.csv` and `data/surfpro_test20.csv` plus `models/pretrained_vae.pt` beneath `MODEL_ROOT`. These training products are external inputs, not distributed weights. See the README encoding caveat.

In [ ]:
RUN_GENERATION = False
MODEL_ROOT = None
if RUN_GENERATION:
    run_experiment("surfactant_generation", data_root=DATA, output=RESULTS / "surfactants/generation", model_root=MODEL_ROOT)